# Exploratory Data Analysis with SQL

**Purpose:** answer reproducible analytical questions about launch sites, orbit categories, and landing outcomes using SQL.

In [1]:
from pathlib import Path
def resolve_data(filename):
    candidates = [Path("data")/filename, Path("../data")/filename]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(filename)

import pandas as pd, sqlite3
df=pd.read_csv(resolve_data("spacex_launch_data.csv"))
con=sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", con, index=False, if_exists="replace")
print("Rows loaded into SQL table:", len(df))


Rows loaded into SQL table: 20


## Query 1 — Launch count by site

In [2]:
pd.read_sql_query(
    "SELECT LaunchSite, COUNT(*) AS Launches FROM SPACEXTBL GROUP BY LaunchSite ORDER BY Launches DESC",
    con
)

,LaunchSite,Launches
0,CCAFS SLC 40,9
1,KSC LC 39A,7
2,VAFB SLC 4E,4


## Query 2 — Orbit frequency and success rate

In [3]:
pd.read_sql_query(
    "SELECT Orbit, COUNT(*) AS Launches, ROUND(AVG(Class),3) AS SuccessRate FROM SPACEXTBL GROUP BY Orbit ORDER BY Launches DESC",
    con
)

,Orbit,Launches,SuccessRate
0,LEO,6,0.833
1,ISS,5,1.000
2,GTO,5,0.600
3,SSO,2,1.000
4,PO,2,0.500


## Query 3 — Landing success by launch site

In [4]:
pd.read_sql_query(
    '''SELECT LaunchSite,
              COUNT(*) AS Total,
              SUM(CASE WHEN Class=1 THEN 1 ELSE 0 END) AS Successful,
              ROUND(AVG(Class)*100,1) AS SuccessPct
       FROM SPACEXTBL
       GROUP BY LaunchSite
       ORDER BY SuccessPct DESC''',
    con
)

,LaunchSite,Total,Successful,SuccessPct
0,KSC LC 39A,7,7,100.0
1,VAFB SLC 4E,4,3,75.0
2,CCAFS SLC 40,9,6,66.7


## Interpretation

SQL makes launch counts and historical success rates auditable directly from the dataset and supports the EDA conclusions used in the presentation.